# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

## 4.02 Árboles Azarosos + Grid Search

Esta versión parte de la lógica del notebook original `z420_ArbolesAzarosos`:

- se usa `rpart` sin perturbar el algoritmo;
- cada árbol se entrena con un subconjunto aleatorio de campos;
- se fija una sola `semilla_primigenia`;
- las probabilidades de los árboles se acumulan;
- la decisión final corresponde al **ensemble completo**.

Para poder comparar hiperparámetros sin subir las 120 combinaciones a Kaggle, se agrega una única partición estratificada 70/30 de `202107`.

La ganancia usada para evaluar el ensemble proviene del notebook de comparación de modelos de la cátedra:

- `+975000` si se envía a un `BAJA+2`;
- `-25000` si se envía incorrectamente;
- `0` si no se envía.

El corte sigue siendo `1/40 = 0.025`.


In [ ]:
PARAM <- list()

# SU semilla primigenia
PARAM$semilla_primigenia <- 149911

# cantidad de árboles del ensemble, igual que en z420
PARAM$num_trees_max <- 32L

# proporción train/validación para evaluar el grid
PARAM$training_pct <- 70L

# función de ganancia de la cátedra
PARAM$ganancia_acierto <- 975000
PARAM$costo_error <- -25000

# grilla de hiperparámetros
PARAM$grid_feature_fraction <- c(0,3, 0.5, 0.7)
PARAM$grid_maxdepth <- c(6, 7, 8, 9)
PARAM$grid_cp <- -1
PARAM$grid_minsplit <- c(400, 600, 800, 1000)
PARAM$grid_ratio_bucket <- c(1/3)

# por seguridad, las submissions no se suben solas
# cambiar a TRUE solamente cuando se quiera subir el TOP 10
PARAM$subir_top10_kaggle <- FALSE


In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")

experimento <- "exp4022_gridsearch_ensemble_original"
dir.create(experimento, showWarnings = FALSE)

setwd(
  paste0(
    "/content/buckets/b1/exp/",
    experimento
  )
)

getwd()


In [ ]:
# lectura del dataset
dataset <- fread(
  "/content/datasets/dataset_pequeno.csv"
)


In [ ]:
# igual que en el z420 original:
# 202107 tiene clase y 202109 es el futuro a predecir

dtrain <- dataset[
  foto_mes == 202107
]

dfuture <- dataset[
  foto_mes == 202109
]

# arreglo clase_ternaria para 202109
dfuture[
  ,
  clase_ternaria := NA
]


In [ ]:
# exactamente la lógica del z420 original:
# todos los campos excepto clase_ternaria pueden ser sorteados

campos_buenos <- copy(
  setdiff(
    colnames(dtrain),
    c("clase_ternaria")
  )
)

length(campos_buenos)


### Partición fija para evaluar el grid

El `z420` original entrena con todo `202107` porque directamente genera submissions para Kaggle.

Para hacer un grid search necesitamos poder medir cada combinación antes de Kaggle. Por eso reservamos una única validación estratificada de 30%.

La partición queda fija para las 120 combinaciones.


In [ ]:
# función de partición utilizada en los notebooks de la cátedra

particionar <- function(
    data,
    division,
    agrupa = "",
    campo = "fold",
    start = 1,
    seed = NA) {

  if (!is.na(seed)) {
    set.seed(seed)
  }

  bloque <- unlist(
    mapply(
      function(x, y) {
        rep(y, x)
      },
      division,
      seq(
        from = start,
        length.out = length(division)
      )
    )
  )

  data[
    ,
    (campo) := sample(
      rep(
        bloque,
        ceiling(.N / length(bloque))
      )
    )[1:.N],
    by = agrupa
  ]
}


In [ ]:
# hago UNA SOLA partición de 202107
# todas las combinaciones usan este mismo train y esta misma validación

datos_grid <- copy(dtrain)

particionar(
  datos_grid,
  division = c(
    PARAM$training_pct,
    100L - PARAM$training_pct
  ),
  agrupa = "clase_ternaria",
  seed = PARAM$semilla_primigenia
)

dtrain_grid <- datos_grid[
  fold == 1
]

dvalid_grid <- datos_grid[
  fold == 2
]

cat(
  "Training grid:",
  nrow(dtrain_grid),
  "\n"
)

cat(
  "Validación grid:",
  nrow(dvalid_grid),
  "\n"
)

datos_grid[
  ,
  .N,
  by = .(
    fold,
    clase_ternaria
  )
][
  order(
    fold,
    clase_ternaria
  )
]


### Evaluar una combinación: un ensemble de 32 Árboles Azarosos

Esta es la parte central.

Para cada combinación de hiperparámetros:

1. se vuelve a fijar `set.seed(PARAM$semilla_primigenia)`;
2. se generan 32 subconjuntos aleatorios de variables;
3. se entrenan 32 `rpart`;
4. se acumulan sus probabilidades de `BAJA+2`;
5. recién después del árbol 32 se calcula `Predicted`;
6. se calcula **una sola ganancia: la ganancia del ensemble**.

El corte se escribe igual que en el notebook original:

`umbral_corte <- (1/40) * cantidad_de_arboles`

sobre la probabilidad acumulada.


In [ ]:
EvaluarEnsemble32 <- function(
  param_rpart,
  feature_fraction
) {

  # acumulador exactamente como en z420
  prob_acumulada <- rep(
    0,
    nrow(dvalid_grid)
  )

  # una sola semilla inicial para toda la secuencia de 32 árboles
  set.seed(
    PARAM$semilla_primigenia
  )

  for (
    arbolito in seq(
      PARAM$num_trees_max
    )
  ) {

    qty_campos_a_utilizar <- as.integer(
      length(campos_buenos) *
        feature_fraction
    )

    # elijo los campos al azar
    campos_random <- sample(
      campos_buenos,
      qty_campos_a_utilizar
    )

    # armo la fórmula igual que en z420
    campos_formula <- paste(
      campos_random,
      collapse = " + "
    )

    formulita <- paste0(
      "clase_ternaria ~ ",
      campos_formula
    )

    # genero el árbol de decisión
    modelo <- rpart(
      formulita,
      data = dtrain_grid,
      xval = 0,
      control = param_rpart
    )

    # aplico sobre la validación
    prediccion <- predict(
      modelo,
      dvalid_grid,
      type = "prob"
    )

    # acumulo probabilidad, igual que el profesor
    prob_acumulada <- (
      prob_acumulada +
        prediccion[, "BAJA+2"]
    )

    if (
      arbolito %% 8 == 0 ||
      arbolito == PARAM$num_trees_max
    ) {

      cat(
        "   árboles:",
        arbolito,
        "de",
        PARAM$num_trees_max,
        "\n"
      )

      flush.console()
    }
  }

  # MISMO criterio del z420 original:
  # para 32 árboles, umbral = (1/40)*32
  umbral_corte <- (
    1 / 40
  ) * PARAM$num_trees_max

  Predicted <- as.numeric(
    prob_acumulada >
      umbral_corte
  )

  # ganancia DEL ENSEMBLE, no promedio de ganancias individuales
  ganancia_valid <- sum(
    ifelse(
      Predicted == 1,
      ifelse(
        dvalid_grid$clase_ternaria == "BAJA+2",
        PARAM$ganancia_acierto,
        PARAM$costo_error
      ),
      0
    )
  )

  # normalizo porque validación representa el 30% de 202107
  proporcion_validacion <- (
    100 - PARAM$training_pct
  ) / 100

  ganancia_ensemble <- (
    ganancia_valid /
      proporcion_validacion
  )

  return(
    list(
      ganancia_ensemble =
        ganancia_ensemble,

      envios =
        sum(Predicted),

      aciertos =
        sum(
          Predicted == 1 &
            dvalid_grid$clase_ternaria ==
              "BAJA+2"
        )
    )
  )
}


## 4.03 Grid Search

Grilla actual:

- `feature_fraction`: 0.5, 0.7, 0.9
- `maxdepth`: 6, 7, 8, 9
- `cp`: -1
- `minsplit`: 200, 400, 600, 800, 1000
- `minbucket`: 1/3 o 0.40 de `minsplit`

Total:

`3 × 4 × 1 × 5 × 2 = 120 combinaciones`

Cada combinación genera un ensemble de 32 árboles:

`120 × 32 = 3840 árboles`


In [ ]:
tb_grid_search <- data.table()

qcombinaciones <- (
  length(PARAM$grid_feature_fraction) *
  length(PARAM$grid_maxdepth) *
  length(PARAM$grid_cp) *
  length(PARAM$grid_minsplit) *
  length(PARAM$grid_ratio_bucket)
)

iter <- 0L

for (
  vmax_depth in PARAM$grid_maxdepth
) {

  for (
    vcp in PARAM$grid_cp
  ) {

    for (
      vmin_split in PARAM$grid_minsplit
    ) {

      for (
        vratio_bucket in PARAM$grid_ratio_bucket
      ) {

        for (
          vfeature_fraction in
            PARAM$grid_feature_fraction
        ) {

          iter <- iter + 1L

          vmin_bucket <- as.integer(
            round(
              vmin_split *
                vratio_bucket
            )
          )

          cat(
            "\nIteración",
            iter,
            "de",
            qcombinaciones,
            "| cp =", vcp,
            "| maxdepth =", vmax_depth,
            "| minsplit =", vmin_split,
            "| minbucket =", vmin_bucket,
            "| feature_fraction =",
            vfeature_fraction,
            "\n"
          )

          flush.console()

          param_rpart <- list(
            cp = vcp,
            minsplit = vmin_split,
            minbucket = vmin_bucket,
            maxdepth = vmax_depth
          )

          resultado <- EvaluarEnsemble32(
            param_rpart =
              param_rpart,

            feature_fraction =
              vfeature_fraction
          )

          # UNA FILA POR COMBINACIÓN
          tb_grid_search <- rbind(
            tb_grid_search,
            data.table(
              iter = iter,
              cp = vcp,
              maxdepth = vmax_depth,
              minsplit = vmin_split,
              minbucket = vmin_bucket,
              ratio_bucket = vratio_bucket,
              feature_fraction =
                vfeature_fraction,
              num_trees =
                PARAM$num_trees_max,
              ganancia_ensemble =
                resultado$ganancia_ensemble,
              envios_ensemble =
                resultado$envios,
              aciertos_ensemble =
                resultado$aciertos
            )
          )

          cat(
            "   ganancia ensemble:",
            round(
              resultado$ganancia_ensemble
            ),
            "\n"
          )

          # guardado incremental
          fwrite(
            tb_grid_search,
            file =
              "gridsearch_ensemble_resultados.csv"
          )

          gc(
            verbose = FALSE
          )
        }
      }
    }
  }
}


## 4.04 Ranking final

Ahora el ranking sí se hace por la misma unidad que después se manda a Kaggle: **el ensemble de 32 árboles**.


In [ ]:
ranking <- tb_grid_search[
  order(
    -ganancia_ensemble
  )
]

ranking[
  1:min(20, .N)
]


In [ ]:
fwrite(
  ranking,
  file =
    "gridsearch_ensemble_ranking.csv"
)

mejor <- ranking[1]

mejor


## 4.05 Generar una submission con una combinación

Esta función vuelve a la lógica exacta del `z420` original:

- usa **todo `202107`**;
- fija una sola vez la `semilla_primigenia`;
- genera 32 árboles con subconjuntos aleatorios de campos;
- acumula probabilidad sobre `202109`;
- usa `umbral_corte = (1/40) * 32`;
- devuelve `numero_de_cliente` y `Predicted`.


In [ ]:
GenerarSubmissionEnsemble32 <- function(
  param_rpart,
  feature_fraction
) {

  tb_prediccion <- dfuture[
    ,
    list(
      numero_de_cliente
    )
  ]

  tb_prediccion[
    ,
    prob_acumulada := 0
  ]

  # igual que en z420:
  # una sola semilla antes del loop
  set.seed(
    PARAM$semilla_primigenia
  )

  for (
    arbolito in seq(
      PARAM$num_trees_max
    )
  ) {

    qty_campos_a_utilizar <- as.integer(
      length(campos_buenos) *
        feature_fraction
    )

    campos_random <- sample(
      campos_buenos,
      qty_campos_a_utilizar
    )

    campos_random <- paste(
      campos_random,
      collapse = " + "
    )

    formulita <- paste0(
      "clase_ternaria ~ ",
      campos_random
    )

    modelo <- rpart(
      formulita,
      data = dtrain,
      xval = 0,
      control = param_rpart
    )

    prediccion <- predict(
      modelo,
      dfuture,
      type = "prob"
    )

    tb_prediccion[
      ,
      prob_acumulada :=
        prob_acumulada +
          prediccion[, "BAJA+2"]
    ]

    if (
      arbolito %% 8 == 0 ||
      arbolito == PARAM$num_trees_max
    ) {

      cat(
        "   árbol:",
        arbolito,
        "de",
        PARAM$num_trees_max,
        "\n"
      )

      flush.console()
    }
  }

  umbral_corte <- (
    1 / 40
  ) * PARAM$num_trees_max

  tb_prediccion[
    ,
    Predicted :=
      as.numeric(
        prob_acumulada >
          umbral_corte
      )
  ]

  return(
    tb_prediccion[
      ,
      list(
        numero_de_cliente,
        Predicted
      )
    ]
  )
}


## 4.06 Generar el TOP 10 para Kaggle

Se generan diez CSV, uno por cada una de las diez mejores combinaciones según `ganancia_ensemble`.

Por seguridad, la subida automática está desactivada inicialmente.  
Si se quiere usar exactamente el mecanismo del notebook original del profesor:

`PARAM$subir_top10_kaggle <- TRUE`


In [ ]:
top10 <- ranking[
  1:min(10, .N)
]

tb_top10_kaggle <- data.table()

for (
  i in seq_len(
    nrow(top10)
  )
) {

  config <- top10[i]

  cat(
    "\n========================================\n",
    "TOP", i,
    "| iter =", config$iter,
    "| ganancia ensemble =",
    round(
      config$ganancia_ensemble
    ),
    "\n",
    "========================================\n"
  )

  param_rpart <- list(
    cp = config$cp,
    minsplit = config$minsplit,
    minbucket = config$minbucket,
    maxdepth = config$maxdepth
  )

  submission <- GenerarSubmissionEnsemble32(
    param_rpart =
      param_rpart,

    feature_fraction =
      config$feature_fraction
  )

  archivo_kaggle <- paste0(
    "KA_TOP",
    sprintf(
      "%.2d",
      i
    ),
    "_iter",
    sprintf(
      "%.3d",
      config$iter
    ),
    ".csv"
  )

  fwrite(
    submission,
    file = archivo_kaggle,
    sep = ","
  )

  cat(
    "Archivo generado:",
    archivo_kaggle,
    "\n"
  )

  # subida opcional, siguiendo el estilo del z420 original
  if (
    PARAM$subir_top10_kaggle
  ) {

    comando <-
      "kaggle competitions submit"

    competencia <-
      "-c labo-1-virtual-inicial"

    arch <- paste(
      "-f",
      archivo_kaggle
    )

    mensaje <- paste0(
      "-m 'TOP", i,
      " cp=", config$cp,
      " minsplit=", config$minsplit,
      " minbucket=", config$minbucket,
      " maxdepth=", config$maxdepth,
      " feature_fraction=",
      config$feature_fraction,
      " ganancia_ensemble=",
      round(
        config$ganancia_ensemble
      ),
      "'"
    )

    linea <- paste(
      comando,
      competencia,
      arch,
      mensaje
    )

    salida <- system(
      linea,
      intern = TRUE
    )

    cat(
      paste(
        salida,
        collapse = "\n"
      ),
      "\n"
    )
  }

  tb_top10_kaggle <- rbind(
    tb_top10_kaggle,
    data.table(
      ranking = i,
      iter = config$iter,
      cp = config$cp,
      maxdepth = config$maxdepth,
      minsplit = config$minsplit,
      minbucket = config$minbucket,
      feature_fraction =
        config$feature_fraction,
      ganancia_ensemble =
        config$ganancia_ensemble,
      archivo =
        archivo_kaggle
    )
  )
}

fwrite(
  tb_top10_kaggle,
  file =
    "top10_ensemble_archivos_kaggle.csv"
)

tb_top10_kaggle


In [ ]:
format(
  Sys.time(),
  "%a %b %d %X %Y"
)
